# Самостоятельная работа
## Исследование гиперпараметров и борьба с переобучением

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import pandas as pd
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

results = []

In [2]:
def get_data(batch_size):
    t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    
    train_full = datasets.CIFAR100(root='./data', train=True, download=True, transform=t)
    test_full = datasets.CIFAR100(root='./data', train=False, download=True, transform=t)
    
    targets = [1, 13, 43]
    
    train_idx = [i for i, l in enumerate(train_full.targets) if l in targets]
    test_idx = [i for i, l in enumerate(test_full.targets) if l in targets]
    
    train_ds = Subset(train_full, train_idx)
    test_ds = Subset(test_full, test_idx)
    
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    )

mapping = {1: 0, 13: 1, 43: 2}

In [3]:
class MLP(nn.Module):
    def __init__(self, h_dim=128, use_dropout=False):
        super().__init__()
        layers = [
            nn.Flatten(),
            nn.Linear(3072, h_dim),
            nn.ReLU()
        ]
        if use_dropout:
            layers.append(nn.Dropout(0.25))
            
        layers.append(nn.Linear(h_dim, 3))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def run_exp(h_dim, lr, bs, ep, name, use_dropout=False, weight_decay=0):
    print(f"Run: {name}")
    train_dl, test_dl = get_data(bs)
    model = MLP(h_dim, use_dropout).to(device)
    
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    
    for e in range(ep):
        model.train()
        tr_corr = 0
        tr_tot = 0
        for x, y in train_dl:
            x, y = x.to(device), torch.tensor([mapping[v.item()] for v in y]).to(device)
            opt.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()
            tr_corr += (out.argmax(1) == y).sum().item()
            tr_tot += y.size(0)
            
    tr_acc = 100 * tr_corr / tr_tot
    
    model.eval()
    ts_corr = 0
    ts_tot = 0
    ts_loss = 0.0
    
    with torch.no_grad():
        for x, y in test_dl:
            x, y = x.to(device), torch.tensor([mapping[v.item()] for v in y]).to(device)
            out = model(x)
            
            loss = loss_fn(out, y)
            ts_loss += loss.item() * x.size(0)
            
            ts_corr += (out.argmax(1) == y).sum().item()
            ts_tot += y.size(0)
            
    ts_acc = 100 * ts_corr / ts_tot
    ts_loss = ts_loss / ts_tot
    
    print(f"Train: {tr_acc:.2f}%, Test Acc: {ts_acc:.2f}%, Test Loss: {ts_loss:.4f}")
    
    results.append({
        'Name': name,
        'LR': lr,
        'Batch': bs,
        'Epochs': ep,
        'Hidden': h_dim,
        'Dropout': 'Yes' if use_dropout else 'No',
        'Train': round(tr_acc, 2),
        'Test Acc': round(ts_acc, 2),
        'Test Loss': round(ts_loss, 4)
    })

### 1. Базовая модель
Здесь мы видим сильное переобучение (Train 100% vs Test ~85%).

In [4]:
run_exp(128, 0.005, 64, 50, "Base")

Run: Base


/Volumes/university/education/7 term/mppr/labs/lab 3/venv/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 100.00%, Test Acc: 84.67%, Test Loss: 0.5567


### 2. Меняем батч

In [5]:
run_exp(128, 0.005, 128, 100, "Big Batch")

Run: Big Batch
Train: 100.00%, Test Acc: 85.67%, Test Loss: 0.5441


### 3. Снижаем LR
Снижение скорости обучения помогает, но переобучение все еще значительно.

In [6]:
run_exp(128, 0.001, 128, 200, "Low LR")

Run: Low LR
Train: 99.33%, Test Acc: 83.33%, Test Loss: 0.4642


### 4. Добавляем регуляризацию (Dropout + L2)
Чтобы устранить переобучение, уменьшаем архитектуру (64 нейрона), добавляем слой Dropout и L2 регуляризацию (weight_decay). Это должно снизить Train Accuracy, но повысить Test Accuracy (уменьшить разрыв).

In [7]:
# Включаем dropout и weight_decay
run_exp(64, 0.001, 128, 200, "Optimized + Reg", use_dropout=True, weight_decay=1e-4)

Run: Optimized + Reg
Train: 97.20%, Test Acc: 86.33%, Test Loss: 0.4284


### Итоговая таблица
Видно, что в последнем эксперименте удалось избавиться от 100% "зазубривания" на трейне и получить лучший результат на тесте.

In [8]:
df = pd.DataFrame(results)

def style_max(s):
    is_max = s == s.max()
    return ['font-weight: bold' if v else '' for v in is_max]

try:
    display(df.style.apply(style_max, subset=['Test Acc']))
except KeyError:
    display(df.style.apply(style_max, subset=['Test']))

,Name,LR,Batch,Epochs,Hidden,Dropout,Train,Test Acc,Test Loss
0,Base,0.005000,64,50,128,No,100.000000,84.670000,0.556700
1,Big Batch,0.005000,128,100,128,No,100.000000,85.670000,0.544100
2,Low LR,0.001000,128,200,128,No,99.330000,83.330000,0.464200
3,Optimized + Reg,0.001000,128,200,64,Yes,97.200000,86.330000,0.428400


### Борьба с переобучением: примененные методы

Для устранения сильного переобучения, наблюдаемого в предыдущих экспериментах (высокая точность на тренировочной выборке при значительно более низкой на тестовой), были применены следующие техники:

1.  **Уменьшение сложности модели:** Количество нейронов в скрытом слое (`hidden_dim`) было уменьшено со 128 до 64. Это снизило "емкость" модели, заставив ее искать более общие закономерности в данных, а не "запоминать" тренировочную выборку.
2.  **Dropout:** В архитектуру нейронной сети добавлен слой `nn.Dropout(0.25)`. Dropout случайным образом "отключает" часть нейронов во время каждой итерации обучения, что не позволяет сети слишком сильно полагаться на отдельные нейроны и делает ее более устойчивой.
3.  **L2-регуляризация (weight_decay):** В оптимизатор `optim.SGD` был добавлен параметр `weight_decay=1e-4`. L2-регуляризация штрафует модель за большие значения весов, тем самым предотвращая их разрастание и способствуя более гладким и обобщающим функциям.

Эти адаптивные меры позволили существенно сократить разрыв между точностью на обучающей и тестовой выборках, улучшив обобщающую способность модели.